# Building ETL Pipelines

**Creating Catalog and Schema**

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS main;
CREATE SCHEMA IF NOT EXISTS main.default;
CREATE VOLUME IF NOT EXISTS main.default.raw_data

**Accessing AWS** 

https://docs.databricks.com/aws/en/connect/unity-catalog/cloud-storage/s3/

-> databricks configure

-> databricks secrets

-> databricks secrets put-secret mySecretScope aws_access_key_id --string-value "ACCESS_KEY_ID"

-> databricks secrets put-secret mySecretScope aws_secret_access_key --string-value

"SECRET_ACCESS_KEY"

> **_NOTE:_** External Cloud Storage connect is not available at databricks Free Edition


**Extracting the data from S3**

In [0]:

# Retrieve secrets from the secret scope
## aws_access_key = dbutils.secrets.get(scope="mySecretSocpe", key = "aws_access_key_id")
## aws_secret_key = dbutils.secrets.get(scope="mySecretSocpe", key = "aws_secret_access_key")

# Set them in Spark configuration for S3 access
## spark.conf.set("fs.s3a.access.key", aws_access_key)
## spark.conf.set("fs.s3.secret.key", aws_secret_key)

# Now you can read from S3 using the s3a:// protocol
## file_path = "s3a://my-bucket/my-file.csv"
## df_emp = spark.read.format('csv').option("header", "true").load(file_path)
## display(df_emp)

**Extracting the Employee data from volume**

In [0]:
file_path = "/Volumes/main/default/raw_data/Employee_data.csv"
df_emp = spark.read.format('csv').option("header", "true").load(file_path)
display(df_emp)

**Filtering and aggregating using Spark**

In [0]:
# Filter: Select employees older than 30
df_filtered = df_emp.filter(df_emp.age > 30)

# Group by: Calculate the average salary per department
df_avg_salary = df_filtered.groupBy('department').agg({'salary': 'avg'})
df_avg_salary.show()

**Filtering and aggregating using SQL**

In [0]:
df_emp.createOrReplaceTempView("employees")

sql_query = """
    SELECT 
        department,
        AVG(salary) as salary
    FROM employees
    WHERE age > 30
    GROUP BY 1
"""

df_avg_salary = spark.sql(sql_query)
df_avg_salary.show()

**Extracting the Department data from volume**

In [0]:
dept_file_path = "/Volumes/main/default/raw_data/departments.csv"
df_dept = spark.read.format('csv').option("header", "true").load(dept_file_path)

display(df_dept)


**Join using SQL**

In [0]:
# Create a temp view
df_dept.createOrReplaceTempView("departments")

# Run SQL query to join employee data with departments 
sql_query = """
    SELECT 
        e.*, 
        d.manager, 
        d.location
    FROM employees e
    INNER JOIN departments d
    ON e.department = d.department
"""
df_joined = spark.sql(sql_query)
df_joined.show()


**Join Using Spark/Python**

In [0]:
df_joined = df_emp.join(df_dept, on = "department", how = "inner") 
df_joined.show()

**Creating the Delta Table**

In [0]:
# Write the joined Dataframe df_joined to a Delta table
# This creates a managed table in Unity Catalog
df_joined.write.mode("overwrite").saveAsTable("main.default.employees_enriched")

# Verify by reading the table
df_loaded = spark.read.table("main.default.employees_enriched")
display(df_loaded)

**Drop CATALOG**

In [0]:
%sql
DROP CATALOG IF EXISTS main CASCADE;

## Notes
---------------

## `df.show()` vs `display(df)`

| | `df.show()` | `display(df)` |
| --- | --- | --- |
| **Origin** | Native PySpark method — works in any Spark environment | Databricks-specific — only works in Databricks notebooks |
| **Output** | Prints plain text rows to stdout (console-style) | Renders a rich interactive HTML table in the notebook |
| **Default rows** | 20 rows | 1,000 rows |
| **Truncation** | Truncates long string values (pass `truncate=False` to disable) | Shows full values with scrollable columns |
| **Interactivity** | None — static text | Interactive: built-in charting, filtering, sorting, and plot views |
| **Return value** | Returns `None` (purely a side-effect action) | Returns the DataFrame object, allowing chaining |

**In short:** Use `display(df)` in Databricks notebooks for rich, interactive output. Use `df.show()` for a quick plain-text dump that works in any Spark environment.

In [0]:
display(df_avg_salary)


In [0]:
df_avg_salary.show()

## `spark.read.format('csv')` vs `spark.read.csv()`

| | `spark.read.format('csv').option(...).load(path)` | `spark.read.csv(path, ...)` |
| --- | --- | --- |
| **Style** | Generic builder pattern — works identically for any format (`parquet`, `json`, `delta`, etc.) | CSV-specific shorthand — dedicated method on `DataFrameReader` |
| **Path** | Passed to `.load(path)` at the end | Passed directly as the first argument |
| **Options** | Chained via `.option("header", "true")` calls | Passed either as `.option()` calls or as key-value kwargs |
| **Verbosity** | More explicit but longer | Shorter and more concise |

Under the hood, `spark.read.csv(path)` is just a convenience wrapper around `spark.read.format('csv').load(path)` — they produce the exact same DataFrame and accept the same options (`header`, `inferSchema`, `delimiter`, `schema`, etc.).

**In short:** Use `.format('csv').load()` for a consistent pattern across file formats or when parameterizing the format. Use `.csv()` for cleaner, more concise code when you know you're always working with CSV.